<a href="https://colab.research.google.com/github/dnlinh2310/CNTT220V1261.1-240270/blob/main/Weather_NhaTrang_2023_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cài đặt các package
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

In [ ]:
# ==============================================================================
# PHẦN 1: GỌI API OPEN-METEO VÀ TẢI DỮ LIỆU
# ==============================================================================
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 12.2451,
    "longitude": 109.1943,
    "start_date": "2023-01-01",
    "end_date": "2026-10-05",
    "daily": [
        "temperature_2m_max",
        "apparent_temperature_max",
        "precipitation_sum",
        "precipitation_hours",
        "sunshine_duration",
        "wind_speed_10m_max",
        "weather_code",
        "temperature_2m_min"
    ],
    "timezone": "Asia/Ho_Chi_Minh"
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]

print(f"📍 Tọa độ: Vĩ độ: {response.Latitude()}°N, Kinh độ: {response.Longitude()}°E")
print(f"🏔️ Độ cao: {response.Elevation()} m")


📍 Tọa độ: Vĩ độ: 12.196836471557617°N, Kinh độ: 109.19892120361328°E
🏔️ Độ cao: 10.0 m


In [ ]:
# Trích xuất các biến số theo ngày
daily = response.Daily()
daily_data = {
    "date": pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left"
    ),
    "Mã Thời Tiết": daily.Variables(6).ValuesAsNumpy(),
    "Nhiệt độ tối đa (2m)": daily.Variables(0).ValuesAsNumpy(),
    "Nhiệt độ cảm nhận tối đa": daily.Variables(1).ValuesAsNumpy(),
    "Tổng lượng mưa": daily.Variables(2).ValuesAsNumpy(),
    "Số giờ mưa": daily.Variables(3).ValuesAsNumpy(),
    "Thời lượng nắng": daily.Variables(4).ValuesAsNumpy(),
    "Tốc độ gió tối đa": daily.Variables(5).ValuesAsNumpy(),
    "Nhiệt độ tối thiểu (2m)": daily.Variables(7).ValuesAsNumpy(),
}

In [ ]:
# xử lí dữ liệu, làm tròn các trường dữ liệu
daily_dataframe = pd.DataFrame(data=daily_data)
daily_dataframe["Nhiệt độ tối đa (2m)"] = daily_dataframe["Nhiệt độ tối đa (2m)"].round(1)
daily_dataframe["Nhiệt độ cảm nhận tối đa"] = daily_dataframe["Nhiệt độ cảm nhận tối đa"].round(1)
daily_dataframe["Nhiệt độ tối thiểu (2m)"] = daily_dataframe["Nhiệt độ tối thiểu (2m)"].round(1)
daily_dataframe["Tổng lượng mưa"] = daily_dataframe["Tổng lượng mưa"].round(1)
daily_dataframe["Tốc độ gió tối đa"] = daily_dataframe["Tốc độ gió tối đa"].round(1)

In [ ]:
csv_filename = "Nha_trang_weather_2023-2026.csv"
daily_dataframe.to_csv(csv_filename, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu dữ liệu thô ra file thành công: {csv_filename}")

✅ Đã lưu dữ liệu thô ra file thành công: Nha_trang_weather_2023-2026.csv


In [ ]:
csv_raw_file = "Nha_trang_weather_2023-2026.csv"
try:
    daily_dataframe.to_csv(csv_raw_file, index=False, encoding="utf-8-sig")
    print(f"✅ Đã lưu dữ liệu thô ra file CSV: {csv_raw_file}")
except PermissionError:
    print(f"❌ LỖI: Hãy đóng file '{csv_raw_file}' đang mở trong Excel/WPS Office rồi chạy lại nhé!")


✅ Đã lưu dữ liệu thô ra file CSV: Nha_trang_weather_2023-2026.csv


In [ ]:
df = daily_dataframe.copy()

In [ ]:
df['date'] = pd.to_datetime(df['date'])
if df['date'].dt.tz is not None:
    df['date'] = df['date'].dt.tz_convert('Asia/Ho_Chi_Minh').dt.tz_localize(None)

# Đặt cột date làm index
df.set_index('date', inplace=True)

# Đổi Thời lượng nắng từ Giây -> Giờ
df['Thời lượng nắng (giờ)'] = df['Thời lượng nắng'] / 3600


In [ ]:
# ==============================================================================
# PHẦN 3: ĐỊNH NGHĨA VÀ LỌC "NGÀY ĐẸP TRỜI" (CÂU 2)
# ==============================================================================
# Tiêu chí chọn lọc du lịch Nha Trang:
# - Nhiệt độ dễ chịu: 24°C <= Temp <= 32°C
# - Tạnh ráo hoàn toàn: Tổng lượng mưa == 0
# - Nắng chiếu >= 6 tiếng/ngày
# - Tốc độ gió < 30 km/h (An toàn di chuyển biển đảo)

In [ ]:
df['Ngày đẹp trời'] = (
    (df['Nhiệt độ tối đa (2m)'] >= 24) & (df['Nhiệt độ tối đa (2m)'] <= 32) &
    (df['Tổng lượng mưa'] == 0) &
    (df['Thời lượng nắng (giờ)'] >= 6) &
    (df['Tốc độ gió tối đa'] < 30)
)

In [ ]:
try:
    resample_freq = 'ME'
    monthly_summary = df.resample(resample_freq).agg(
        Nhiệt_độ_Max_TB=('Nhiệt độ tối đa (2m)', 'mean'),
        Nhiệt_độ_Min_TB=('Nhiệt độ tối thiểu (2m)', 'mean'),
        Tổng_lượng_mưa=('Tổng lượng mưa', 'sum'),
        Số_ngày_đẹp_trời=('Ngày đẹp trời', 'sum'),
        Tổng_số_ngày=('Ngày đẹp trời', 'count')
    )
except ValueError:
    resample_freq = 'M'
    monthly_summary = df.resample(resample_freq).agg(
        Nhiệt_độ_Max_TB=('Nhiệt độ tối đa (2m)', 'mean'),
        Nhiệt_độ_Min_TB=('Nhiệt độ tối thiểu (2m)', 'mean'),
        Tổng_lượng_mưa=('Tổng lượng mưa', 'sum'),
        Số_ngày_đẹp_trời=('Ngày đẹp trời', 'sum'),
        Tổng_số_ngày=('Ngày đẹp trời', 'count')
    )

In [ ]:
monthly_summary['Nhiệt_độ_Max_TB'] = monthly_summary['Nhiệt_độ_Max_TB'].round(1)
monthly_summary['Nhiệt_độ_Min_TB'] = monthly_summary['Nhiệt_độ_Min_TB'].round(1)
monthly_summary['Tổng_lượng_mưa'] = monthly_summary['Tổng_lượng_mưa'].round(1)
monthly_summary['Tỷ_lệ_ngày_đẹp_(%)'] = (monthly_summary['Số_ngày_đẹp_trời'] / monthly_summary['Tổng_số_ngày'] * 100).round(1)

In [ ]:
print("\n=== BẢNG TỔNG HỢP RESAMPLE THEO THÁNG (12 THÁNG ĐẦU TIÊN) ===")
print(monthly_summary.head(12))

csv_resampled_file = "Nha_trang_monthly_summary.csv"
monthly_summary.to_csv(csv_resampled_file, encoding="utf-8-sig")
print(f"\n✅ Đã xuất bảng tổng hợp resample ra file: {csv_resampled_file}")


=== BẢNG TỔNG HỢP RESAMPLE THEO THÁNG (12 THÁNG ĐẦU TIÊN) ===
            Nhiệt_độ_Max_TB  Nhiệt_độ_Min_TB  Tổng_lượng_mưa  \
date                                                           
2023-01-31        25.400000        21.700001      192.600006   
2023-02-28        26.799999        21.799999       41.200001   
2023-03-31        28.400000        22.000000       17.700001   
2023-04-30        31.400000        24.600000       21.299999   
2023-05-31        32.799999        26.000000       64.599998   
2023-06-30        33.500000        26.600000       79.300003   
2023-07-31        32.200001        25.900000       81.900002   
2023-08-31        33.200001        26.100000      114.900002   
2023-09-30        31.900000        25.700001      113.000000   
2023-10-31        29.799999        24.900000      151.600006   
2023-11-30        28.200001        23.700001      203.000000   
2023-12-31        27.400000        23.100000      113.699997   

            Số_ngày_đẹp_trời  Tổng_số_ng

In [ ]:
# ==============================================================================
# PHẦN 5: TỔNG HỢP KẾT QUẢ VÀ TRẢ LỜI CÂU HỎI BÀI TẬP
# ==============================================================================
monthly_pattern = df.groupby(df.index.month).agg(
    Nhiệt_độ_TB=('Nhiệt độ tối đa (2m)', 'mean'),
    Lượng_mưa_TB=('Tổng lượng mưa', 'mean'),
    Tổng_số_ngày_đẹp=('Ngày đẹp trời', 'sum')
)


In [ ]:
num_years = len(df.index.year.unique())
monthly_pattern['Ngày_đẹp_trời_TB'] = (monthly_pattern['Tổng_số_ngày_đẹp'] / num_years).round(1)

print("\n=== QUY LUẬT THỜI TIẾT TRUNG BÌNH 12 THÁNG TRONG NĂM (NHA TRANG) ===")
print(monthly_pattern[['Nhiệt_độ_TB', 'Lượng_mưa_TB', 'Ngày_đẹp_trời_TB']])

best_month = monthly_pattern['Ngày_đẹp_trời_TB'].idxmax()
avg_nice_days = monthly_pattern.loc[best_month, 'Ngày_đẹp_trời_TB']

print(f"\n🌟 KẾT LUẬN CÂU 2: Tháng có số ngày đẹp trời nhiều nhất là THÁNG {best_month} (Trung bình {avg_nice_days} ngày đẹp trời/tháng).")


=== QUY LUẬT THỜI TIẾT TRUNG BÌNH 12 THÁNG TRONG NĂM (NHA TRANG) ===
      Nhiệt_độ_TB  Lượng_mưa_TB  Ngày_đẹp_trời_TB
date                                             
1       26.242743      2.111290              13.0
2       27.501770      1.549558               9.2
3       28.987097      0.781452              13.8
4       31.709999      0.415833              10.5
5       32.671776      2.308871               0.5
6       33.221668      2.086667               0.5
7       33.199997      2.589516               0.2
8       33.421772      2.734678               1.0
9       31.537500      6.105000               0.8
10      29.825510      6.042857               1.0
11      28.222221     10.863334               2.5
12      27.134409      7.491398               2.2

🌟 KẾT LUẬN CÂU 2: Tháng có số ngày đẹp trời nhiều nhất là THÁNG 3 (Trung bình 13.8 ngày đẹp trời/tháng).


In [ ]:
from google.colab import files

# Tải file dữ liệu thời tiết thô
files.download('Nha_trang_weather_2023-2026.csv')

# # Tải file bảng tổng hợp theo tháng
# files.download('Nha_trang_monthly_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Tải file bảng tổng hợp theo tháng
files.download('Nha_trang_monthly_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>